# Giai đoạn 01 — Chuẩn bị nguồn & Cắt theo tác phẩm (`CPO_09`, `CPO_10`, `CPO_11`, `CPO_12`)

Notebook này gộp bốn script chuẩn bị nguồn trước đây (`cat_bac_su.py`, `cat_hich_tuong_si.py`, `cat_ly_tran.py`, `cat_van_sach.py`) thành **một tài liệu duy nhất**, tách **mỗi tác phẩm thành một cell riêng** và cho phép **người dùng tự nhập số trang / khoảng trang**.

## Hướng dẫn sử dụng nhanh

1. **Chạy Cell 1 (Khởi tạo & Helper)** trước để nạp `ten_file.py` và các module bảo vệ trong `cong_cu/` (`Book`, `Source`, `Box`, `Workspace`, `pdf_ops`, `ui`).
2. **Chuyển xuống đúng cell của tác phẩm cần xử lý**:
   - **Bắc sứ thông lục (`CPO_09`)** — Cell 3
   - **Hịch tướng sĩ (`CPO_10`)** — Cell 5
   - **Thơ văn Lý–Trần, tập 3 (`CPO_12`)** — Cell 7
   - **Văn sách thi Đình Thăng Long–Hà Nội, tập 2 (`CPO_11`)** — Cell 9
3. **Tự nhập số trang / khoảng trang** bằng một trong hai cách:
   - Sửa trực tiếp biến `CUSTOM_PAGES` ngay trong cell (ví dụ `"49-52, 55, 60-61"` hoặc `((187, 366),)` hoặc `None` để dùng mặc định phân công); **hoặc**
   - Đặt `PROMPT_INPUT_PAGES = True` để hộp thoại `input()` hiện ra cho bạn gõ tay lúc chạy cell.
4. **Chọn chế độ `ACTION` trong cell của tác phẩm**:
   - `"info"`: xem thông tin file nguồn, số trang PDF và khoảng trang đang chọn (chỉ đọc).
   - `"prepare"`: tạo cấu trúc thư mục `01_nguon/` + `05_quan_ly/` và sao lưu nguyên vẹn PDF gốc.
   - `"check"`: kiểm tra SHA-256 của bản sao nguồn và các vùng đã lưu.
   - `"summary"`: xem bảng đủ/thiếu 4 bản văn (`han`, `dich_am`, `dich_nghia`, `dich_tho`) của `WORK_ID`.
   - `"crop_page"`: cắt trực tiếp 1 vùng hoặc loạt trang bạn nhập (`PAGE_TO_CROP`, `BOX = (x0, y0, x1, y1)`).
   - `"text_lines"`: cắt đoạn từ TXT UTF-8 có sẵn (`TEXT_FILE`, `LINE_START`, `LINE_END`).
   - `"gui"`: mở cửa sổ giao diện Tkinter (`open_window`) để kéo chuột khoanh vùng như trước đây.

> **Nguyên tắc bảo toàn nguồn (Bước 3):** Không sửa/di chuyển file gốc trong `data/`, không tự chạy OCR hay Text Cleaning tại đây, và không ghép nhầm 4 bản văn (`Hán`, `Dịch âm`, `Dịch nghĩa`, `Dịch thơ`).

In [ ]:
import json
from pathlib import Path
import re
import sys

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from ten_file import CAC_BAN, DPI, MA_TAC_PHAM, OUTPUT_ROOT, TEN_NGUON
from cong_cu import pdf_ops
from cong_cu.models import Book, Box, Source
from cong_cu.workspace import Workspace

print("Project root:", ROOT)
print("Output root:", OUTPUT_ROOT)


def parse_page_ranges(spec, default_ranges):
    """Chuyển chuỗi/tuple số trang người dùng nhập (vd: '187-190, 195') thành tuple các cặp (start, end)."""
    if spec is None:
        return tuple(default_ranges)
    if isinstance(spec, str):
        cleaned = spec.strip()
        if not cleaned:
            return tuple(default_ranges)
        ranges = []
        for part in re.split(r"[,;]+", cleaned):
            token = part.strip()
            if not token:
                continue
            if "-" in token:
                a, b = token.split("-", 1)
                start, end = int(a.strip()), int(b.strip())
            else:
                start = end = int(token)
            if start < 1 or end < start:
                raise ValueError(f"Khoảng trang không hợp lệ: {token!r}")
            ranges.append((start, end))
        return tuple(ranges)
    if isinstance(spec, int):
        return ((spec, spec),)
    return tuple((int(a), int(b)) for a, b in spec)


def prompt_user_pages(label, current_spec, default_ranges):
    raw = input(f"[{label}] Nhập số trang hoặc khoảng trang (vd: 49-55, 60 | Enter giữ {current_spec or default_ranges}): ")
    return parse_page_ranges(raw if raw.strip() else current_spec, default_ranges)


def run_book_cell(
    book: Book,
    *,
    action: str = "info",
    source_key: str | None = None,
    work_id: str | None = None,
    kind: str = "han",
    version: int = 1,
    page_to_crop: int | None = None,
    box: tuple[float, float, float, float] = (0.0, 0.0, 1.0, 1.0),
    dpi: int = DPI,
    extract_text: bool = False,
    text_file: Path | str | None = None,
    line_start: int | None = None,
    line_end: int | None = None,
    note: str = "",
    allow_outside_scope: bool = False,
):
    """Thực thi một thao tác chuẩn bị nguồn cho Book với cấu hình và khoảng trang của cell."""
    ws = Workspace(book)
    target_source = source_key or book.sources[0].key
    target_work = work_id or book.first_work

    if action == "info":
        print("===", book.code, "—", book.title, "===")
        print("Ghi chú:", book.note)
        for src in book.sources:
            exists = Path(src.path).is_file()
            pdf_meta = pdf_ops.info(src.path) if exists else {"file_exists": False}
            print(f" - Nguồn [{src.key}] ({src.default_kind}): {src.path} | phạm vi={src.ranges} | {pdf_meta}")
            if src.note:
                print(f"   Ghi chú nguồn: {src.note}")
        return None
    if action == "prepare":
        sources = ws.prepare()
        print(f"Đã tạo/kiểm tra {ws.root} ({len(sources)} bản sao nguồn).")
        return sources
    if action == "check":
        res = ws.check()
        print(json.dumps(res, ensure_ascii=False, indent=2))
        return res
    if action == "summary":
        res = ws.summary(target_work, version)
        print(json.dumps(res, ensure_ascii=False, indent=2))
        return res
    if action == "crop_page":
        if page_to_crop is None:
            raise ValueError("Cần đặt PAGE_TO_CROP (số trang PDF gốc >= 1) khi ACTION='crop_page'.")
        res = ws.save_pdf_region(
            target_source,
            int(page_to_crop),
            Box(*box),
            target_work,
            kind,
            version,
            dpi,
            extract_text,
            note,
            allow_outside_scope,
        )
        print("Đã có, bỏ qua" if res.get("skipped") else "Đã lưu", res["work"], res["kind"], res["sequence"])
        return res
    if action == "text_lines":
        if not text_file or line_start is None or line_end is None:
            raise ValueError("Cần đặt TEXT_FILE, LINE_START, LINE_END khi ACTION='text_lines'.")
        res = ws.save_text_region(Path(text_file).resolve(), target_work, kind, version, line_start=int(line_start), line_end=int(line_end), note=note)
        print("Đã lưu TXT:", res["work"], res["kind"], res["sequence"])
        return res
    if action == "gui":
        from cong_cu.ui import open_window
        open_window(ws)
        return None
    raise ValueError(f"ACTION không hợp lệ: {action!r}. Chọn info | prepare | check | summary | crop_page | text_lines | gui.")

---
## 1. Bắc sứ thông lục (`CPO_09`)
*(Gộp từ `cat_bac_su.py`)*
- Hai nguồn riêng (`viet` PDF 32–61 bàn giao TXT thô; `han` PDF 49–101 cắt ảnh vùng lớn) — **không giả định cùng số trang là bản dịch tương ứng**.
- Mốc đối chiếu tham khảo: `{"loi_tua": (9, 2), "loi_de_tu": (11, 4), "quyen_1": (13, 7), "quyen_4": (93, 55)}`.
- Lưu ý: Ảnh Hán nộp bỏ PDF `57, 58, 63, 64, 71, 72, 79, 80, 91, 92` theo yêu cầu người dùng; bản PDF gốc giữ nguyên.

In [ ]:
# === CẤU HÌNH BẮC SỨ THÔNG LỤC (CPO_09) ===
PROMPT_INPUT_PAGES = False   # Đổi thành True nếu muốn nhập số trang trực tiếp qua input()
CUSTOM_VIET_PAGES = None     # Ví dụ: "32-61" hoặc "32-54, 55-61" | None = mặc định (32, 61)
CUSTOM_HAN_PAGES = None      # Ví dụ: "49-56, 59-62" hoặc "49-101" | None = mặc định (49, 101)

ACTION = "info"              # info | prepare | check | summary | crop_page | text_lines | gui
SOURCE_KEY = "han"           # "han" hoặc "viet"
WORK_ID = "doan0001"
KIND = "han"                 # "han" hoặc "dich_nghia"
VERSION = 1
PAGE_TO_CROP = 49            # Số trang PDF gốc nếu chọn ACTION = "crop_page"
BOX = (0.0, 0.0, 1.0, 1.0)   # Khung cắt chuẩn hóa 0..1 (x0, y0, x1, y1)
ALLOW_OUTSIDE_SCOPE = False

MOC_DOI_CHIEU = {"loi_tua": (9, 2), "loi_de_tu": (11, 4), "quyen_1": (13, 7), "quyen_4": (93, 55)}
viet_ranges = prompt_user_pages("Bắc sứ Việt", CUSTOM_VIET_PAGES, ((32, 61),)) if PROMPT_INPUT_PAGES else parse_page_ranges(CUSTOM_VIET_PAGES, ((32, 61),))
han_ranges = prompt_user_pages("Bắc sứ Hán", CUSTOM_HAN_PAGES, ((49, 101),)) if PROMPT_INPUT_PAGES else parse_page_ranges(CUSTOM_HAN_PAGES, ((49, 101),))

BOOK_BAC_SU = Book(
    code=MA_TAC_PHAM["bac_su"],
    title="Bắc sứ thông lục",
    sources=(
        Source("viet", TEN_NGUON["bac_su_viet"], viet_ranges, "viet", "PDF Text, bàn giao TXT nguồn thô: bắt đầu sau mục 21 ở PDF 32; kết thúc đúng câu được giao tại PDF 61. Giữ chú thích và ký tự nguồn, chưa làm sạch."),
        Source("han", TEN_NGUON["bac_su_han"], han_ranges, "han", "Theo bảng mới phancong.md: PDF 49–101. PDF gốc giữ đủ trang; ảnh nộp bỏ PDF 57, 58, 63, 64, 71, 72, 79, 80, 91, 92 theo yêu cầu người dùng."),
    ),
    first_work="doan0001",
    work_range=None,
    note="Chuẩn bị nguồn TV2 theo phancong.md: Việt PDF Text 32–61 bàn giao TXT thô; Hán PDF 49–101 cắt ảnh vùng lớn. "
         "Việt 32 lấy sau mục 21; Việt 61 dừng sau câu để tôi thêm vui vẻ cảm kích. "
         "Không tách từng cột/đoạn, không dóng hàng, không mở rộng sang Hán 102.",
)

run_book_cell(
    BOOK_BAC_SU,
    action=ACTION,
    source_key=SOURCE_KEY,
    work_id=WORK_ID,
    kind=KIND,
    version=VERSION,
    page_to_crop=PAGE_TO_CROP,
    box=BOX,
    allow_outside_scope=ALLOW_OUTSIDE_SCOPE,
)

---
## 2. Hịch tướng sĩ — mẫu chung (`CPO_10`)
*(Gộp từ `cat_hich_tuong_si.py`)*
- Mẫu chung kiểm tra, không tính thêm vào bộ cặp riêng của thành viên.
- Dịch nghĩa PDF `4–6`; Hán & dịch âm xen kẽ PDF `11–20`; diễn thơ PDF `21–26`. Không lấy tiếng Anh (`27–30`).

In [ ]:
# === CẤU HÌNH HỊCH TƯỚNG SĨ (CPO_10) ===
PROMPT_INPUT_PAGES = False   # Đổi thành True nếu muốn nhập số trang qua input()
CUSTOM_PAGES = None          # Ví dụ: "4-6, 11-26" hoặc "11-20" | None = mặc định ((4, 6), (11, 26))

ACTION = "info"              # info | prepare | check | summary | crop_page | text_lines | gui
WORK_ID = "hich_tuong_si"
KIND = "han"                 # han | dich_am | dich_nghia | dich_tho
VERSION = 1
PAGE_TO_CROP = 11
BOX = (0.0, 0.0, 1.0, 1.0)
EXTRACT_TEXT = False
ALLOW_OUTSIDE_SCOPE = False

hich_ranges = prompt_user_pages("Hịch tướng sĩ", CUSTOM_PAGES, ((4, 6), (11, 26))) if PROMPT_INPUT_PAGES else parse_page_ranges(CUSTOM_PAGES, ((4, 6), (11, 26)))

BOOK_HICH = Book(
    code=MA_TAC_PHAM["hich"],
    title="Hịch tướng sĩ — mẫu chung",
    sources=(Source("chung", TEN_NGUON["hich"], hich_ranges),),
    first_work="hich_tuong_si",
    work_range=None,
    note="Mẫu chung do TV1 quản lý, TV2 kiểm tra; không tính thêm một bộ cặp. "
         "Dịch nghĩa PDF 4–6; Hán và dịch âm xen kẽ PDF 11–20; diễn thơ PDF 21–26. "
         "Không lấy tiếng Anh PDF 27–30. Chú thích riêng PDF 7–10 để đối chiếu.",
)

run_book_cell(
    BOOK_HICH,
    action=ACTION,
    work_id=WORK_ID,
    kind=KIND,
    version=VERSION,
    page_to_crop=PAGE_TO_CROP,
    box=BOX,
    extract_text=EXTRACT_TEXT,
    allow_outside_scope=ALLOW_OUTSIDE_SCOPE,
)

---
## 3. Thơ văn Lý–Trần, tập 3 (`CPO_12`)
*(Gộp từ `cat_ly_tran.py`)*
- Phạm vi mặc định: bài `131–207`, PDF `187–366`.
- Trang 187 gồm phần cuối bài 130 ở phía trên và đầu bài 131 ở phía dưới; Bài 207 có Hán ở PDF `365`, dịch nghĩa ở PDF `366`.

In [ ]:
# === CẤU HÌNH THƠ VĂN LÝ–TRẦN (CPO_12) ===
PROMPT_INPUT_PAGES = False   # Đổi thành True nếu muốn nhập số trang qua input()
CUSTOM_PAGES = None          # Ví dụ: "187-190" hoặc "187, 365-366" | None = mặc định ((187, 366),)
CUSTOM_WORK_RANGE = (131, 207)

ACTION = "info"              # info | prepare | check | summary | crop_page | text_lines | gui
WORK_ID = "bai0131"
KIND = "han"                 # han | dich_am | dich_nghia | dich_tho
VERSION = 1
PAGE_TO_CROP = 187
BOX = (0.0, 0.0, 1.0, 1.0)
ALLOW_OUTSIDE_SCOPE = False

ly_tran_ranges = prompt_user_pages("Thơ văn Lý–Trần", CUSTOM_PAGES, ((187, 366),)) if PROMPT_INPUT_PAGES else parse_page_ranges(CUSTOM_PAGES, ((187, 366),))

BOOK_LY_TRAN = Book(
    code=MA_TAC_PHAM["ly_tran"],
    title="Thơ văn Lý–Trần, tập 3",
    sources=(Source("chung", TEN_NGUON["ly_tran"], ly_tran_ranges),),
    first_work="bai0131",
    work_range=CUSTOM_WORK_RANGE,
    note="TV2: bài 131–207. PDF 187 chứa cuối bài 130 của TV1 và đầu bài 131 của bạn. "
         "Chọn đúng bài; gom đủ Hán, dịch âm, dịch nghĩa, dịch thơ qua các trang. "
         "Bài 207: Hán ở PDF 365, dịch nghĩa ở PDF 366.",
)

run_book_cell(
    BOOK_LY_TRAN,
    action=ACTION,
    work_id=WORK_ID,
    kind=KIND,
    version=VERSION,
    page_to_crop=PAGE_TO_CROP,
    box=BOX,
    allow_outside_scope=ALLOW_OUTSIDE_SCOPE,
)

---
## 4. Văn sách thi Đình Thăng Long–Hà Nội, tập 2 (`CPO_11`)
*(Gộp từ `cat_van_sach.py`)*
- Phạm vi mặc định: bài `14–18`, PDF `188–365`.
- Mốc tham khảo từng bài (`MOC_BAI`): Bài 14 (`188–220`, nghĩa `197`, Hán `210`), Bài 15 (`221–253`, `229`/`244`), Bài 16 (`254–300`, `266`/`285`), Bài 17 (`301–333`, `309`/`324`), Bài 18 (`334–365`, `343`/`356`).

In [ ]:
# === CẤU HÌNH VĂN SÁCH THI ĐÌNH (CPO_11) ===
PROMPT_INPUT_PAGES = False   # Đổi thành True nếu muốn nhập số trang qua input()
CUSTOM_PAGES = None          # Ví dụ: "188-220" cho bài 14 | None = mặc định ((188, 365),)
CUSTOM_WORK_RANGE = (14, 18)

ACTION = "info"              # info | prepare | check | summary | crop_page | text_lines | gui
WORK_ID = "bai0014"
KIND = "han"                 # han | dich_am | dich_nghia | dich_tho
VERSION = 1
PAGE_TO_CROP = 210
BOX = (0.0, 0.0, 1.0, 1.0)
ALLOW_OUTSIDE_SCOPE = False

MOC_BAI = {
    14: {"trang": (188, 220), "dich_nghia": 197, "han": 210},
    15: {"trang": (221, 253), "dich_nghia": 229, "han": 244},
    16: {"trang": (254, 300), "dich_nghia": 266, "han": 285},
    17: {"trang": (301, 333), "dich_nghia": 309, "han": 324},
    18: {"trang": (334, 365), "dich_nghia": 343, "han": 356},
}
van_sach_ranges = prompt_user_pages("Văn sách Thi Đình", CUSTOM_PAGES, ((188, 365),)) if PROMPT_INPUT_PAGES else parse_page_ranges(CUSTOM_PAGES, ((188, 365),))

BOOK_VAN_SACH = Book(
    code=MA_TAC_PHAM["van_sach"],
    title="Văn sách thi Đình Thăng Long–Hà Nội, tập 2",
    sources=(Source("chung", TEN_NGUON["van_sach"], van_sach_ranges),),
    first_work="bai0014",
    work_range=CUSTOM_WORK_RANGE,
    note="TV2: bài 14–18 / PDF 188–365. Bài 14: dịch nghĩa bắt đầu 197, Hán bắt đầu 210; "
         "bài 15: 229/244; bài 16: 266/285; bài 17: 309/324; bài 18: 343/356. "
         "Gom từng bài, không coi toàn bộ khoảng trang là một loại văn bản.",
)

run_book_cell(
    BOOK_VAN_SACH,
    action=ACTION,
    work_id=WORK_ID,
    kind=KIND,
    version=VERSION,
    page_to_crop=PAGE_TO_CROP,
    box=BOX,
    allow_outside_scope=ALLOW_OUTSIDE_SCOPE,
)

---
## Lưu ý bảo toàn dữ liệu
- File PDF đầu vào trong `data/` và bản sao trong `01_nguon/*/ban_goc/` được bảo vệ bằng SHA-256 và chỉ đọc.
- Thứ tự lưu vùng sẽ là thứ tự các file `phanXXXX_pdfpYYYY.png` trong thư mục tác phẩm (`<MA_SACH>_<TV>_<WORK>_<KIND>_vXX/`).
- Bước này chỉ xuất PNG vùng cắt hoặc TXT trích xuất thô (nếu bật `extract_text`), không tự động chuyển sang OCR hay Text Cleaning.